# Day 04 Tutorial — Performance Tuning Essentials

**Goal:** Plans, shuffle, partitioning, cache.


### Environment setup
Skip pip install on Databricks/Fabric. Locally you may need: `pip install pyspark pandas`.


In [ ]:
# %pip install pyspark==3.5.1 pandas -q


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName('AzureDE-InterviewPrep')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark


In [ ]:
df = spark.range(0, 100000).withColumn('grp', (F.col('id') % 10).cast('int'))
df2 = (
    spark.range(0, 100000)
    .withColumnRenamed('id', 'id2')
    .withColumn('grp', (F.col('id2') % 10).cast('int'))
)


## Explain plans


In [ ]:
df.join(df2, 'grp').explain('formatted')


## Repartition vs coalesce


In [ ]:
print('partitions', df.rdd.getNumPartitions())
wide = df.repartition(8, 'grp')
narrow = wide.coalesce(2)
print(wide.rdd.getNumPartitions(), narrow.rdd.getNumPartitions())


## Cache


In [ ]:
base = df.filter(F.col('id') < 50000)
base.cache()
print(base.count())
print(base.groupBy('grp').count().count())
base.unpersist()


## Interview checklist
- Shuffle is expensive network movement.
- repartition shuffles; coalesce mainly reduces partitions.
- Debug via Spark UI, explain, skew/spill signals.
